# Extração de Features de Áudio — Wav2Vec 2.0
Pipeline: .mp4 → ffmpeg → .wav → Wav2Vec 2.0 → embeddings .npy

In [1]:
# CÉLULA 0: Instalar dependências
!pip install transformers librosa soundfile

'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [1]:
# CÉLULA 1: Imports e configuração
import os
import re
import time
import subprocess
import numpy as np
import torch
import librosa
import soundfile as sf
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm.notebook import tqdm

# === AJUSTE ESSES PATHS ===
BASE_DIR = r'C:\Users\ddonz\OneDrive\Documentos\Aislan\data'
VIDEOS_DIR = os.path.join(BASE_DIR, 'Videos')        # pasta com vídeos .mp4
AUDIO_DIR = os.path.join(BASE_DIR, 'audio_wav')       # .wav extraídos (temp)
OUTPUT_DIR = os.path.join(BASE_DIR, 'audio_features') # embeddings .npy

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Videos dir: {VIDEOS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
Videos dir: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\Videos
Output dir: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\audio_features


In [2]:
# CÉLULA 2: Verificar ffmpeg e estrutura de vídeos

# Testar ffmpeg
try:
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print("ffmpeg OK:", result.stdout.split('\n')[0])
except FileNotFoundError:
    print("ERRO: ffmpeg não encontrado!")
    print("Instale: https://www.gyan.dev/ffmpeg/builds/ (Windows)")
    print("Ou: conda install ffmpeg")

# Contar vídeos
all_videos = []
for pid in sorted(os.listdir(VIDEOS_DIR)):
    visit_path = os.path.join(VIDEOS_DIR, pid, 'Visite_1')
    if not os.path.isdir(visit_path):
        continue
    for vfile in sorted(os.listdir(visit_path)):
        if vfile.endswith('.mp4'):
            all_videos.append({
                'pid': pid,
                'video_name': vfile,
                'video_path': os.path.join(visit_path, vfile)
            })

print(f"\nTotal vídeos .mp4: {len(all_videos)}")
print(f"Participantes: {len(set(v['pid'] for v in all_videos))}")
print(f"Exemplo: {all_videos[0]['video_path']}")

ffmpeg OK: ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers

Total vídeos .mp4: 1427
Participantes: 300
Exemplo: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\Videos\82553\Visite_1\82553_Question_1_2024-08-22_12-11-55_Video.mp4


In [3]:
# CÉLULA 3: Extrair áudio dos vídeos (.mp4 → .wav 16kHz mono)

extracted = 0
skipped = 0
errors_audio = []

for v in tqdm(all_videos, desc="Extraindo áudio"):
    pid_dir = os.path.join(AUDIO_DIR, v['pid'])
    os.makedirs(pid_dir, exist_ok=True)
    
    wav_name = v['video_name'].replace('.mp4', '.wav')
    wav_path = os.path.join(pid_dir, wav_name)
    
    # Resume: pular se já existe
    if os.path.exists(wav_path):
        skipped += 1
        continue
    
    try:
        # ffmpeg: extrair áudio em 16kHz mono (formato Wav2Vec)
        cmd = [
            'ffmpeg', '-i', v['video_path'],
            '-vn',              # sem vídeo
            '-acodec', 'pcm_s16le',  # PCM 16-bit
            '-ar', '16000',     # 16kHz (Wav2Vec requer)
            '-ac', '1',         # mono
            '-y',               # sobrescrever
            wav_path
        ]
        subprocess.run(cmd, capture_output=True, check=True)
        extracted += 1
    except Exception as e:
        errors_audio.append(f"{v['pid']}/{v['video_name']}: {str(e)}")

print(f"\nExtraídos: {extracted} | Já existiam: {skipped} | Erros: {len(errors_audio)}")
if errors_audio:
    for e in errors_audio[:5]:
        print(f"  - {e}")

Extraindo áudio:   0%|          | 0/1427 [00:00<?, ?it/s]


Extraídos: 1427 | Já existiam: 0 | Erros: 0


In [4]:
# CÉLULA 4: Carregar modelo Wav2Vec 2.0

print("Carregando Wav2Vec 2.0 base...")
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
wav2vec_model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(device)
wav2vec_model.eval()

print(f"Modelo carregado no {device}")
print(f"Hidden size: {wav2vec_model.config.hidden_size}")  # 768

Carregando Wav2Vec 2.0 base...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

C:\Users\ddonz\miniconda3\envs\bah\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ddonz\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo carregado no cuda
Hidden size: 768


In [5]:
# CÉLULA 5: Extrair embeddings de áudio

MAX_AUDIO_SECS = 60  # limite pra evitar OOM em vídeos longos
SAMPLE_RATE = 16000

errors_embed = []
processed = 0
skipped = 0
start_time = time.time()

for v in tqdm(all_videos, desc="Extraindo embeddings"):
    pid_out_dir = os.path.join(OUTPUT_DIR, v['pid'])
    os.makedirs(pid_out_dir, exist_ok=True)
    
    npy_name = v['video_name'].replace('.mp4', '.npy')
    npy_path = os.path.join(pid_out_dir, npy_name)
    
    # Resume
    if os.path.exists(npy_path):
        skipped += 1
        continue
    
    # Carregar .wav
    wav_name = v['video_name'].replace('.mp4', '.wav')
    wav_path = os.path.join(AUDIO_DIR, v['pid'], wav_name)
    
    if not os.path.exists(wav_path):
        errors_embed.append(f"{v['pid']}/{v['video_name']}: wav não encontrado")
        continue
    
    try:
        # Carregar áudio
        audio, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
        
        # Truncar se muito longo
        max_samples = MAX_AUDIO_SECS * SAMPLE_RATE
        if len(audio) > max_samples:
            audio = audio[:max_samples]
        
        # Processar com Wav2Vec
        inputs = processor(
            audio, 
            sampling_rate=SAMPLE_RATE, 
            return_tensors='pt',
            padding=True
        ).input_values.to(device)
        
        with torch.no_grad():
            outputs = wav2vec_model(inputs)
            # hidden_states: (1, T, 768) — T ~ audio_length / 320 (~20ms por frame)
            embeddings = outputs.last_hidden_state.squeeze(0).cpu().numpy()
        
        # Salvar
        np.save(npy_path, embeddings.astype(np.float32))  # float32 pra economizar espaço
        processed += 1
        
    except Exception as e:
        errors_embed.append(f"{v['pid']}/{v['video_name']}: {str(e)}")
    
    # Progresso a cada 100 vídeos
    if (processed + skipped) % 200 == 0 and processed > 0:
        elapsed = time.time() - start_time
        rate = processed / elapsed
        remaining = len(all_videos) - processed - skipped
        eta = remaining / rate / 60 if rate > 0 else 0
        print(f"  {processed} processados | {rate:.1f} vídeos/s | ETA: {eta:.1f} min")

elapsed = time.time() - start_time
print(f"\nConcluído em {elapsed/60:.1f} min")
print(f"Processados: {processed} | Já existiam: {skipped} | Erros: {len(errors_embed)}")
if errors_embed:
    print("Primeiros erros:")
    for e in errors_embed[:10]:
        print(f"  - {e}")

Extraindo embeddings:   0%|          | 0/1427 [00:00<?, ?it/s]

  200 processados | 7.4 vídeos/s | ETA: 2.8 min
  400 processados | 9.9 vídeos/s | ETA: 1.7 min
  600 processados | 11.0 vídeos/s | ETA: 1.2 min
  800 processados | 11.9 vídeos/s | ETA: 0.9 min
  1000 processados | 12.6 vídeos/s | ETA: 0.6 min
  1200 processados | 13.1 vídeos/s | ETA: 0.3 min
  1400 processados | 13.5 vídeos/s | ETA: 0.0 min

Concluído em 1.8 min
Processados: 1427 | Já existiam: 0 | Erros: 0


In [6]:
# CÉLULA 6: Validação

saved_files = []
for pid in os.listdir(OUTPUT_DIR):
    pid_dir = os.path.join(OUTPUT_DIR, pid)
    if not os.path.isdir(pid_dir):
        continue
    for f in os.listdir(pid_dir):
        if f.endswith('.npy'):
            fpath = os.path.join(pid_dir, f)
            arr = np.load(fpath)
            saved_files.append((pid, f, arr.shape))

print(f"Arquivos .npy salvos: {len(saved_files)}")
print(f"Participantes: {len(set(p for p, _, _ in saved_files))}")

# Estatísticas
timesteps = [s[0] for _, _, s in saved_files]
dims = set(s[1] for _, _, s in saved_files)
print(f"Timesteps por vídeo: min={min(timesteps)}, max={max(timesteps)}, "
      f"média={np.mean(timesteps):.0f}")
print(f"Dimensão embedding: {dims}")
print(f"\nExemplo: {saved_files[0][1]} → shape {saved_files[0][2]}")

# Checar NaN
nan_count = 0
for pid, fname, _ in saved_files[:100]:
    arr = np.load(os.path.join(OUTPUT_DIR, pid, fname))
    if np.isnan(arr).any():
        nan_count += 1
print(f"NaN nos primeiros 100: {nan_count}")

Arquivos .npy salvos: 1427
Participantes: 300
Timesteps por vídeo: min=156, max=2999, média=1306
Dimensão embedding: {768}

Exemplo: 82553_Question_1_2024-08-22_12-11-55_Video.npy → shape (834, 768)
NaN nos primeiros 100: 0
